# Distance Travelled

## Objective

Travel between venues can affect preparation and recovery. In this notebook I match every team-game row in `FullTeamGames` to the venue from that same team's previous game in the same tournament, then calculate the distance travelled between those two host cities.

For each team's first game in a tournament there is no previous match venue. I use the 25th percentile of non-zero, non-first-game travel distances within that tournament. This is conservative because teams usually have more time than average to get to their first stadium, but setting travel to 0 would be incorrect and would distort the model.

# Inputs

- `1.DataCleaning-R/Data/RDS/FullTeamGames.rds`

# Output

- `1.DataCleaning-R/Data/RDS/DistanceTravelled.rds`

In [7]:
library(tidyverse)
library(tidygeocoder)
library(here)

Lets get the World Cup host cities in coordinates using `tidygeocoder`, following the same approach as the weather notebook. I include host country in the geocode address so cities resolve to the correct tournament host context.

In [8]:
FullTeamGames <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "FullTeamGames.rds"))

host_countries <- tibble(
  tournament_id = c("WC-2010", "WC-2014", "WC-2018", "WC-2022"),
  host_country = c("South Africa", "Brazil", "Russia", "Qatar")
)

CityCoords <- FullTeamGames %>%
  distinct(tournament_id, city_name) %>%
  left_join(host_countries, by = "tournament_id") %>%
  mutate(geocode_address = paste(city_name, host_country, sep = ", ")) %>%
  distinct(city_name, host_country, geocode_address) %>%
  geocode(geocode_address, method = "osm", lat = lat, long = long)

CityCoords %>%
  arrange(host_country, city_name) %>%
  print(n = Inf)

Passing 37 addresses to the Nominatim single address geocoder



Next I define a haversine helper to calculate great-circle distance in kilometers between consecutive venues.

In [ ]:
haversine_km <- function(lat1, lon1, lat2, lon2) {
  radius_km <- 6371
  to_rad <- pi / 180

  d_lat <- (lat2 - lat1) * to_rad
  d_lon <- (lon2 - lon1) * to_rad
  lat1 <- lat1 * to_rad
  lat2 <- lat2 * to_rad

  a <- sin(d_lat / 2)^2 + cos(lat1) * cos(lat2) * sin(d_lon / 2)^2
  c <- 2 * atan2(sqrt(a), sqrt(1 - a))

  radius_km * c
}

I order each team within each tournament by match date and time, lag the previous city, calculate the venue-to-venue distance, and then fill first-game distances with the tournament-specific 25th percentile of non-zero travel distances.

In [ ]:
GamesWithCoords <- FullTeamGames %>%
  left_join(
    CityCoords %>% select(city_name, lat, long),
    by = "city_name"
  )

DistanceRaw <- GamesWithCoords %>%
  arrange(tournament_id, team_id, match_date, match_time, match_id) %>%
  group_by(tournament_id, team_id) %>%
  mutate(
    previous_city_name = lag(city_name),
    previous_lat = lag(lat),
    previous_long = lag(long),
    first_game_in_tournament = is.na(previous_city_name),
    distance_traveled_km = haversine_km(previous_lat, previous_long, lat, long)
  ) %>%
  ungroup()

first_game_distances <- DistanceRaw %>%
  filter(!first_game_in_tournament, distance_traveled_km > 0) %>%
  group_by(tournament_id) %>%
  summarise(
    first_game_distance_km = quantile(distance_traveled_km, probs = 0.25, na.rm = TRUE),
    .groups = "drop"
  )

DistanceTravelled <- DistanceRaw %>%
  left_join(first_game_distances, by = "tournament_id") %>%
  mutate(
    distance_traveled_km = if_else(
      first_game_in_tournament,
      first_game_distance_km,
      distance_traveled_km
    )
  ) %>%
  select(distance_traveled_km, match_id, team_id)

head(DistanceTravelled)

distance_traveled_km,match_id,team_id
<dbl>,<chr>,<chr>
304.0729,M-2010-06,T-01
1546.3260,M-2010-23,T-01
1308.6457,M-2010-38,T-01
304.0729,M-2010-04,T-03
0.0000,M-2010-18,T-03
292.5333,M-2010-35,T-03


Lets manually check the first-game imputation values and confirm the final output has one row per team-game.

In [ ]:
first_game_distances

DistanceTravelled %>%
  summarise(
    rows = n(),
    missing_distance = sum(is.na(distance_traveled_km)),
    min_distance = min(distance_traveled_km),
    median_distance = median(distance_traveled_km),
    max_distance = max(distance_traveled_km)
  )

set.seed(2026)
DistanceTravelled %>%
  slice_sample(n = 10)

tournament_id,first_game_distance_km
<chr>,<dbl>
WC-2010,304.07294
WC-2014,622.27277
WC-2018,781.13224
WC-2022,35.54345


rows,missing_distance,min_distance,median_distance,max_distance
<int>,<int>,<dbl>,<dbl>,<dbl>
512,0,0,622.2728,2978.304


distance_traveled_km,match_id,team_id
<dbl>,<chr>,<chr>
622.27277,M-2014-51,T-46
95.24529,M-2010-50,T-83
35.54345,M-2022-03,T-65
781.13224,M-2018-02,T-26
100.64379,M-2010-58,T-32
1308.64574,M-2010-56,T-73
781.13224,M-2018-01,T-62
304.07294,M-2010-16,T-73
489.69041,M-2014-49,T-13


Looks good.

In [ ]:
saveRDS(DistanceTravelled, here("1.DataCleaning-R", "Data", "RDS", "DistanceTravelled.rds"))